# Concept steering di Llama-3.1-8B con CAA

**Concept injection** (stesso meccanismo di Golden Gate Claude / Eiffel Tower Llama / brand
steering) su tre concetti: una **marca** (Nike), un **personaggio** (Albert Einstein),
un **luogo** (il Colosseo).

Si studia come l'effetto dello steering cambi al variare di **layer di iniezione** e
**intensità (multiplier)**. Metrica: frazione di generazioni (su prompt neutri) in cui il
modello finisce per parlare del concetto.

> **Eseguire su Google Colab / Kaggle (GPU T4, 16 GB).** Runtime -> Change runtime type -> GPU.


## 0. Setup


In [ ]:
!pip -q install transformers accelerate bitsandbytes matplotlib

In [ ]:
# Clona la TUA repo per importare il package `caa` e i dataset.
REPO_URL = 'https://github.com/<tuo-utente>/caa-llama-steering.git'
!git clone $REPO_URL 2>/dev/null || echo 'Repo gia presente o URL da impostare'

import sys, os, glob
ROOT = 'caa-llama-steering' if os.path.isdir('caa-llama-steering/caa') else '.'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, 'data')
RESULTS = os.path.join(ROOT, 'results'); os.makedirs(RESULTS, exist_ok=True)

In [ ]:
# Modello gated: accetta la licenza su huggingface.co/meta-llama/Llama-3.1-8B-Instruct
# e incolla un token (Settings -> Access Tokens).
from huggingface_hub import login
HF_TOKEN = ''  # <-- token HF
if HF_TOKEN:
    login(HF_TOKEN)

## 1. Carica il modello (4-bit)


In [ ]:
import torch, matplotlib.pyplot as plt
from caa import (load_model, ActivationSteerer, build_steering_vectors,
                 concept_mention_rate, generate, load_concept_dataset,
                 build_concept_pairs, NEUTRAL_PROMPTS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_model(load_in_4bit=True, hf_token=HF_TOKEN or None)
N_LAYERS = len(model.model.layers)
print('Layer totali:', N_LAYERS)

## 2. Carica i tre concetti e costruisci i vettori di steering

Per ogni concetto: coppie contrastive (frase sul concetto vs frase neutra) ->
`v_L = media(pos) - media(neg)` su un insieme di layer candidati.


In [ ]:
CONCEPT_FILES = {
    'Nike':      os.path.join(DATA, 'concept_brand_nike.json'),
    'Einstein':  os.path.join(DATA, 'concept_person_einstein.json'),
    'Colosseo':  os.path.join(DATA, 'concept_place_colosseo.json'),
}

candidate_layers = list(range(2, N_LAYERS, 4))  # campiona alcuni layer
steerer = ActivationSteerer(model)

concepts = {}  # nome -> {'keywords':..., 'vectors':{layer: vec}}
for name, path in CONCEPT_FILES.items():
    ds = load_concept_dataset(path)
    pairs = build_concept_pairs(tokenizer, ds['pairs'])
    vecs = build_steering_vectors(model, tokenizer, steerer, pairs, candidate_layers, device)
    concepts[name] = {'keywords': ds['keywords'], 'vectors': vecs}
    print(f"{name}: {len(pairs)} coppie, vettori per layer {candidate_layers}")

## 3. Baseline (senza steering)

Tasso di menzione 'spontaneo' del concetto sui prompt neutri: dovrebbe essere ~0.


In [ ]:
steerer.remove_hooks()
baseline = {name: concept_mention_rate(model, tokenizer, NEUTRAL_PROMPTS, c['keywords'], device)
            for name, c in concepts.items()}
print('Baseline mention rate:', baseline)

## Esperimento 1 — Sweep del multiplier (layer fisso)

A layer intermedio fisso, alziamo l'intensità: ci aspettiamo che il tasso di menzione cresca
fino a 'saturare', e che a valori troppo alti il testo degeneri.


In [ ]:
FIXED_LAYER = candidate_layers[len(candidate_layers)//2]  # layer centrale
multipliers = [0, 2, 4, 6, 8, 10, 12]
print('Layer fisso:', FIXED_LAYER)

rate_by_mult = {name: [] for name in concepts}
for name, c in concepts.items():
    for m in multipliers:
        if m == 0:
            steerer.remove_hooks()
        else:
            steerer.set_steering({FIXED_LAYER: c['vectors'][FIXED_LAYER]}, multiplier=m)
        r = concept_mention_rate(model, tokenizer, NEUTRAL_PROMPTS, c['keywords'], device)
        rate_by_mult[name].append(r)
    steerer.remove_hooks()
    print(name, rate_by_mult[name])

In [ ]:
plt.figure(figsize=(7,4))
for name in concepts:
    plt.plot(multipliers, rate_by_mult[name], 'o-', label=name)
plt.xlabel('Multiplier (alpha)'); plt.ylabel('Tasso di menzione del concetto')
plt.title(f'Effetto del multiplier @ layer {FIXED_LAYER}')
plt.ylim(-0.05, 1.05); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'exp1_multiplier_sweep.png'), dpi=150); plt.show()

## Esperimento 2 — Sweep del layer (multiplier fisso)

Stesso multiplier su layer diversi: mostra in quale parte della rete l'iniezione è più efficace
(tipicamente i layer intermedi).


In [ ]:
FIXED_MULT = 8
rate_by_layer = {name: [] for name in concepts}
for name, c in concepts.items():
    for l in candidate_layers:
        steerer.set_steering({l: c['vectors'][l]}, multiplier=FIXED_MULT)
        r = concept_mention_rate(model, tokenizer, NEUTRAL_PROMPTS, c['keywords'], device)
        rate_by_layer[name].append(r)
    steerer.remove_hooks()
    print(name, rate_by_layer[name])

In [ ]:
plt.figure(figsize=(8,4))
for name in concepts:
    plt.plot(candidate_layers, rate_by_layer[name], 's-', label=name)
plt.xlabel('Layer di iniezione'); plt.ylabel('Tasso di menzione del concetto')
plt.title(f'Effetto del layer @ multiplier {FIXED_MULT}')
plt.ylim(-0.05, 1.05); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'exp2_layer_sweep.png'), dpi=150); plt.show()

## Esperimento 3 — Generazione libera (qualitativa)

La parte più 'vistosa' per la presentazione: lo stesso prompt neutro, prima senza steering
e poi con il concetto iniettato. Cambia il concetto modificando `demo`.


In [ ]:
demo = 'Colosseo'   # prova anche 'Nike' o 'Einstein'
prompt = 'Tell me about your typical day.'
c = concepts[demo]
for m in [0, FIXED_MULT]:
    if m == 0:
        steerer.remove_hooks()
    else:
        steerer.set_steering({FIXED_LAYER: c['vectors'][FIXED_LAYER]}, multiplier=m)
    print(f'\n===== {demo} | multiplier = {m} =====')
    print(generate(model, tokenizer, prompt, device, max_new_tokens=150))
steerer.remove_hooks()

## 4. Conclusioni (da compilare)

- Qual è il layer più efficace per ciascun concetto? I layer intermedi confermano la teoria?
- Da quale multiplier il modello inizia a 'fissarsi' sul concetto? Quando il testo degrada?
- Differenze tra i tre tipi di concetto (marca / persona / luogo): qualcuno è più facile da iniettare?
- Coerenza tra metrica quantitativa (tasso di menzione) e impressione qualitativa.
- Limiti: dimensione dei dataset, scelta delle keyword, quantizzazione 4-bit, posizione del token.
